# 01 - Random Forest Baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfefGuesmi/msdd-arab-youth-space-hackathon/blob/main/MSDD-Net/notebooks/01_Random_Forest_Baseline.ipynb)

Trains the three MARIDA-paper Random Forest baselines and prints their per-class results.

Runs on CPU in 10 to 20 minutes. No GPU needed.

## 1. Prepare environment

If you already ran notebook 00, skip to the last line of this cell.

In [ ]:
import os
if not os.path.isdir('msdd-arab-youth-space-hackathon'):
    !git clone -b msddnet https://github.com/AfefGuesmi/msdd-arab-youth-space-hackathon.git

%cd msdd-arab-youth-space-hackathon/MSDD-Net
!pip install -q -r requirements.txt
!python -c 'import sklearn, joblib, rasterio; print("deps ok")'

## 2. Train and evaluate on all three feature sets

In [ ]:
# Bands only (RF_SS)
!python train_rf.py --features ss --data_path data --predict_patches True

In [ ]:
# Bands + spectral indices (RF_SS+SI)
!python train_rf.py --features ss_si --data_path data --predict_patches True

In [ ]:
# Bands + indices + GLCM (RF_SS+SI+GLCM) -- best in the MARIDA paper
!python train_rf.py --features ss_si_glcm --data_path data --predict_patches True

## 3. Load results and print a comparison table

In [ ]:
import json, os
import pandas as pd

rows = []
for fs in ('ss', 'ss_si', 'ss_si_glcm'):
    p = f'trained_models/rf_{fs}/results.json'
    if not os.path.isfile(p):
        continue
    with open(p) as f:
        r = json.load(f)
    for split in ('val', 'test'):
        if split not in r:
            continue
        m = r[split]
        rows.append({
            'features': fs,
            'split': split,
            'MD IoU': m['MD']['iou'],
            'MD F1': m['MD']['f1'],
            'macro IoU': m['Average']['iou'],
            'macro F1': m['Average']['f1'],
            'pixel acc': m['pixel_accuracy'],
        })

df = pd.DataFrame(rows)
df

## 4. Top-10 feature importances for the full feature set

In [ ]:
import json
with open('trained_models/rf_ss_si_glcm/results.json') as f:
    r = json.load(f)

print('Top 10 features by Gini importance:')
for name, imp in r['feature_importance'][:10]:
    print(f'  {name:10s} {imp:.4f}')

## Next steps

The RF predictions are now saved in `trained_models/rf_ss_si_glcm/predictions/`. They are used by notebook 04 to plot RF predictions alongside U-Net and MSDD-Net.